In [4]:
!pip install sam3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 13.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 165.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=0de5648284c105d4e9dd571eafcd38deaebdbacc65521b514be903317896932d
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built iopath


In [3]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-yhzft0q3
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-yhzft0q3
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=026076bc73799e133ddc5873d1c835bebf36ceae1f2eae8f4d96230fcebbb43b
  Stored in directory: /tmp/pip-ephem-wheel-cache-l73ky55c/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything


In [7]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

--2026-03-12 16:15:15--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.171.22.68, 3.171.22.33, 3.171.22.118, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.171.22.68|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G   224MB/s    in 11s     

2026-03-12 16:15:26 (220 MB/s) - ‘sam_vit_h_4b8939.pth’ saved [2564550879/2564550879]



In [26]:
# 1. Install SAM dependencies
!pip install timm git+https://github.com/facebookresearch/segment-anything.git

# 2. Clone your project from GitHub
import os
repo_url = "https://github.com/JoonParrrk/ME_592_robotics_HW.git"
repo_name = "ME_592_robotics_HW"

if not os.path.exists(repo_name):
    print("Cloning project from ground up...")
    !git clone {repo_url}
else:
    print("Project already exists, pulling latest changes...")
    %cd {repo_name}
    !git pull
    %cd ..

# 3. Enter the specific assignment folder
%cd {repo_name}/robotic-grasping

# 4. Verify the files are there
!ls

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-pe1jtkck
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-pe1jtkck
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
Project already exists, pulling latest changes...
/content/ME_592_robotics_HW
Already up to date.
/content
[Errno 2] No such file or directory: 'ME_592_robotics_HW/robotic-grasping'
/content
ME_592_robotics_HW  sample_data  sam_vit_h_4b8939.pth


In [27]:
import torch
import sys

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print("✅ SUCCESS: Connected to Colab GPU")
    print(f"🤖 GPU Model: {device_name}")
    print(f"💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ STILL ON CPU: Check the top-right Kernel setting!")

# Double check we are in the cloud environment
print(f"🌐 Platform: {sys.platform}")

✅ SUCCESS: Connected to Colab GPU
🤖 GPU Model: NVIDIA RTX PRO 6000 Blackwell Server Edition
💾 VRAM Available: 101.97 GB
🌐 Platform: linux


In [28]:
import torch
from segment_anything import sam_model_registry, SamPredictor
import numpy as np
from PIL import Image

# 1. SETUP THE POWERHOUSE
device = "cuda"
model_type = "vit_h" # Use the HUGE model, you have the memory for it!
checkpoint = "sam_vit_h_4b8939.pth"

sam = sam_model_registry[model_type](checkpoint=checkpoint).to(device)
predictor = SamPredictor(sam)

# 2. SELECTION API
# This is how we tell the GPU to process your specific vegetable images
def run_segmentation(image_np, box_coords):
    predictor.set_image(image_np) # The GPU pre-calculates the image features here
    
    # Box is [x_min, y_min, x_max, y_max]
    input_box = np.array(box_coords)
    
    masks, scores, _ = predictor.predict(
        box=input_box,
        multimask_output=False
    )
    return masks[0], scores[0]


In [29]:
!find /content/ -name "bpe_simple_vocab_16e6.txt.gz"

/content/ME_592_robotics_HW/bpe_simple_vocab_16e6.txt.gz


In [30]:
# UPDATE THESE based on the 'find' results above
VOCAB_PATH = "/content/temp_repo/bpe_simple_vocab_16e6.txt.gz"
CHECKPOINT_PATH = "/content/temp_repo/sam3.pt"
IMG_PATH = "/content/temp_repo/robotic-grasping/6_test_images/pcd0802r.png"

In [31]:
import os

# Check if we can see the vocab file from our current position
vocab_exists = os.path.exists("../bpe_simple_vocab_16e6.txt.gz")
print(f"Can Python see the vocab file? {vocab_exists}")

# Check if we can see the image
image_exists = os.path.exists("6_test_images/pcd0802r.png")
print(f"Can Python see the image? {image_exists}")

if vocab_exists and image_exists:
    print("🚀 All systems go! Run the SAM 3 inference now.")
else:
    print("❌ Paths are still mismatched. Check '!ls ..' to see where files are.")

Can Python see the vocab file? True
Can Python see the image? False
❌ Paths are still mismatched. Check '!ls ..' to see where files are.


In [32]:
import os
import sys

# 1. DEFINE ABSOLUTE PATHS (The Linux way)
# This assumes your repo is at /content/ME_592_robotics_HW/
BASE_DIR = "/content/ME_592_robotics_HW"
PROJECT_ROOT = os.path.join(BASE_DIR, "robotic-grasping")

# Define all paths relative to the BASE_DIR
VOCAB_PATH = os.path.join(BASE_DIR, "bpe_simple_vocab_16e6.txt.gz")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "sam3.pt")
IMG_PATH = os.path.join(PROJECT_ROOT, "6_test_images/pcd0802r.png")

# 2. FIX THE MODULE IMPORT
# We add the BASE_DIR to the path so 'import sam3' works regardless of where we are
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print(f"Checking for Vocab: {os.path.exists(VOCAB_PATH)}")
print(f"Checking for Model: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Checking for Image: {os.path.exists(IMG_PATH)}")

# ==========================================
# 3. LOAD THE MODEL 
# ==========================================
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SAM 3 onto {device}...")

model = build_sam3_image_model(
    checkpoint_path=CHECKPOINT_PATH,
    bpe_path=VOCAB_PATH
).to(device)

processor = Sam3Processor(model)

Checking for Vocab: True
Checking for Model: False
Checking for Image: False
Loading SAM 3 onto cuda...


FileNotFoundError: [Errno 2] No such file or directory: '/content/ME_592_robotics_HW/sam3.pt'